In [118]:
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = "Your Api Key"

In [119]:
!pip install -q \
youtube-transcript-api \
langchain-community \
langchain \
langchain-huggingface \
sentence-transformers \
huggingface_hub \
faiss-cpu \
tiktoken \
python-dotenv \
requests==2.32.4

In [120]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound
)

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEndpoint,ChatHuggingFace,HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [121]:
video_id = "Gfr50f6ZBvo"

try:
    transcript_list = YouTubeTranscriptApi().fetch(
        video_id,
        languages=['en']
    )

    # Convert to list of dicts
    transcript = [
        {
            "text": chunk.text,
            "start": chunk.start,
            "duration": chunk.duration
        }
        for chunk in transcript_list
    ]

    transcript

except TranscriptsDisabled:
    print("Transcripts are disabled for this video")

except NoTranscriptFound:
    print("No transcript found")

In [122]:
transcript

[{'text': 'the following is a conversation with',
  'start': 0.08,
  'duration': 3.44},
 {'text': 'demus hasabis', 'start': 1.76, 'duration': 4.96},
 {'text': 'ceo and co-founder of deepmind', 'start': 3.52, 'duration': 5.119},
 {'text': 'a company that has published and builds',
  'start': 6.72,
  'duration': 4.48},
 {'text': 'some of the most incredible artificial',
  'start': 8.639,
  'duration': 4.561},
 {'text': 'intelligence systems in the history of',
  'start': 11.2,
  'duration': 4.8},
 {'text': 'computing including alfred zero that',
  'start': 13.2,
  'duration': 3.68},
 {'text': 'learned', 'start': 16.0, 'duration': 2.96},
 {'text': 'all by itself to play the game of gold',
  'start': 16.88,
  'duration': 4.559},
 {'text': 'better than any human in the world and',
  'start': 18.96,
  'duration': 5.6},
 {'text': 'alpha fold two that solved protein',
  'start': 21.439,
  'duration': 4.241},
 {'text': 'folding', 'start': 24.56, 'duration': 4.16},
 {'text': 'both tasks consider

Step 1b - Indexing (Text Splitting)

In [123]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Convert transcript list -> text
transcript_text = " ".join(
    item["text"] for item in transcript
)

chunks = splitter.create_documents([transcript_text])

In [124]:
print(len(chunks))

168


In [125]:
chunks[100]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

Step 1c & 1b - Indexing(Embedding Generation and Storing in Vector Store)

In [117]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [126]:
vector_store.index_to_docstore_id

{0: '2ecb51c8-e02c-4308-9b4a-fa1c5a8ae1ef',
 1: '128eb650-1ab9-4d07-924c-3bf3fb67cfd7',
 2: 'b840e8cb-9ab5-4045-94e1-28eaa49d62c9',
 3: '73296ac3-1f46-45bd-b55e-96eb6fc97a10',
 4: 'f34ed39b-e7ad-44c9-a0e3-f28f7a5bc0c3',
 5: 'ad915b9a-f642-4978-a47f-8e3f93ad848c',
 6: 'dea7cf32-8e7d-4b18-a5eb-ada4bceebd54',
 7: '74034e02-aacf-48e0-9e45-661b02c9e54e',
 8: '35da9550-15d6-4662-a1cd-1084d436b6b6',
 9: 'ebd8ae84-850a-4400-ba30-83a987e9d44d',
 10: 'dfcfacba-16c6-4922-a710-13ee489bdfbb',
 11: 'dca39e8d-3d27-4f29-a5c5-2c92bfc71221',
 12: 'a8df9278-9f08-40e1-8eda-fbfc26084ed5',
 13: 'a439b1a4-96c3-481b-82e0-219b3dd4d97b',
 14: 'a7a19e04-e44a-482f-876d-d893d20b1af1',
 15: 'd5e09a3d-93a6-482c-b8f6-9c46fb17170d',
 16: '051d0b49-b1e7-4af0-8215-d24df841141c',
 17: '3588323f-21ef-4a2c-a728-1d352ce767da',
 18: 'd89d5cdc-980b-4cd2-8344-7eab91b7bc18',
 19: '217c4840-9461-43fc-8903-78052fb5e529',
 20: '5099d71c-8da6-4856-abf7-5c4107ea93ed',
 21: '16867cdd-5132-4b06-aaa0-478f19b06099',
 22: 'c7ad39c2-ab55-

In [127]:
vector_store.get_by_ids(['5bbcba00-1b04-4979-a3fd-e59617f9e44c'])

[]

Step 2 - Retrieval


In [128]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs={"k": 4})

In [129]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7aae3ec7e540>, search_kwargs={'k': 4})

In [130]:
retriever.invoke("What is deepmind?")

[Document(id='c21050cc-e859-4820-b2f7-3557c0c6c14e', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

Step 3 - Augmentation

In [131]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [132]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [133]:
retrieved_docs

[Document(id='95f326ad-6002-4f45-ae4b-aef18845e86c', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [134]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

In [135]:
final_prompt = prompt.format(context=context_text, question=question)
final_prompt

"\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which 

Step 4 - Generation

In [136]:
# model that can generate multiple queries for a given input question

model = ChatHuggingFace(llm=HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V4-Pro",
    task="text-generation",

))

In [137]:
answer = model.invoke(final_prompt)
print(answer.content)

Yes, nuclear fusion is discussed. The speaker mentions collaborating with EPFL (the Swiss technical institute) to use their test reactor, identifying bottleneck problems in fusion that AI methods can help with, and publishing a Nature paper on using deep reinforcement learning for magnetic control of tokamak plasmas. They explain that they were able to hold plasma in specific shapes for a record amount of time using an AI controller. They are now talking to fusion startups to find the next problem to tackle in this area.


Building a Chain

In [138]:
from langchain_core.runnables import RunnablePassthrough,RunnableParallel,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [139]:
def fromat_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [140]:
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(fromat_docs),
    "question": RunnablePassthrough()
})

In [141]:
parallel_chain.invoke("Who is Demis?")

{'context': "to get world peace because there's also other corrupting things like wanting power over people and this kind of stuff which is not necessarily satisfied by by just abundance but i think it will help um and i think uh but i think ultimately ai is not going to be run by any one person or one organization i think it should belong to the world belong to humanity um and i think maybe many there'll be many ways this will happen and ultimately um everybody should have a say in that do you have advice for uh young people in high school and college maybe um if they're interested in ai or interested in having a big impact on the world what they should do to have a career they can be proud of her to have a life they can be proud of i love giving talks to the next generation what i say to them is actually two things i i think the most important things to learn about and to find out about when you're when you're young is what are your true passions is first of all there's two things on

In [142]:
parser = StrOutputParser()

In [143]:
main_chain = parallel_chain | prompt | model | parser

In [145]:
main_chain.invoke('Can you summarize the video in detail?')

'I can only summarize based on the transcript excerpt provided, which does not cover the full video. From this fragment, the conversation covers several ideas:\n\n- There is a desire for a deeper, simpler explanation of physics that goes beyond the Standard Model, which the speakers feel is incomplete. This hypothetical explanation would address long-standing mysteries like consciousness, life, and gravity.\n- Lex Fridman then thanks his guest, “Damas” (likely Demis Hassabis), calling him a special human being and expressing honor at the conversation.\n- After a sponsor mention and a quote from Edsger Dijkstra, the discussion turns to intelligence. One sign of intelligence is the ability to explain complex topics simply, referencing Richard Feynman’s view. If an AI couldn’t explain something simply, it might indicate a lack of intelligence.\n- There is a note on human-AI symbiosis, with humans already blending with their devices like phones.\n- Creativity isn’t born in a vacuum; great 